# The FIELDS Instrument Suite for Solar Probe Plus — Implementation / 구현

**Paper**: S.D. Bale et al., "The FIELDS Instrument Suite for Solar Probe Plus", Space Sci. Rev. 204, 49–82 (2016). DOI: 10.1007/s11214-016-0244-5

This notebook reproduces three core measurement physics from the FIELDS paper:
1. V1-V4 double-probe E-field reconstruction with synthetic Alfvén-wave fluctuations near 10 R_s.
2. Fluxgate and search-coil magnetometer transfer functions and noise floor (Fig. 12, Fig. 2).
3. DC-to-MHz wave dispersion relations (Alfvén, ion-cyclotron, whistler, Langmuir) at SPP perihelion plasma parameters.

이 노트북은 FIELDS 논문의 세 핵심 측정 물리를 재현합니다:
1. 10 R_s 근처 합성 알펜파 요동에 대한 V1-V4 이중탐침 전기장 재구성.
2. Fluxgate와 search-coil 자력계의 전달 함수와 노이즈 한계(Fig. 12, Fig. 2).
3. SPP 근일점 플라즈마 매개변수에서 DC ~ MHz 파동 분산 관계(알펜, 이온-사이클로트론, 휘슬러, 랭뮤어).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Physical constants (SI) / 물리 상수
EPS0 = 8.8541878128e-12   # vacuum permittivity, F/m
MU0 = 1.25663706212e-6    # vacuum permeability, H/m
ME = 9.1093837015e-31     # electron mass, kg
MP = 1.67262192369e-27    # proton mass, kg
QE = 1.602176634e-19      # elementary charge, C
KB = 1.380649e-23         # Boltzmann constant, J/K
C_LIGHT = 2.99792458e8    # speed of light, m/s
EV_TO_J = QE              # 1 eV = 1.602e-19 J

## Part 1: V1-V4 Double-Probe E-Field Reconstruction / V1-V4 이중탐침 전기장 재구성

**Concept / 개념**: FIELDS measures three electric field components from five voltage sensors:
$$
E_{12} = (V_1 - V_2)/L_{12}, \quad E_{34} = (V_3 - V_4)/L_{34}, \quad E_z = (V_5 - \bar V)/L_z
$$
where $\bar V = (V_1+V_2+V_3+V_4)/4$ removes the common-mode spacecraft floating potential.

FIELDS는 다섯 개 전압 센서로 세 성분 전기장을 측정합니다. 평균값 $\bar V$를 빼는 연산으로 우주선 부유 전위를 제거합니다.

We simulate Alfvénic E-field fluctuations at 10 R_s ($\delta E \sim 100$ mV/m, see Table 2) on a 6-7 m baseline antenna pair, contaminated by spacecraft floating potential drifts and uncorrelated sensor noise. We then verify that differential reconstruction recovers the true field while rejecting common-mode contamination.

Table 2의 10 R_s 알펜 E-field 요동 ($\delta E \sim 100$ mV/m)을 6-7 m 기준선 안테나 쌍에 모사하고, 우주선 전위 변동과 상관 없는 센서 잡음을 더한 뒤, 차동 재구성이 진짜 신호를 회복하면서 공통모드를 제거하는지 검증합니다.

In [ ]:
def simulate_fields_voltages(t, e_true, baseline_12, baseline_34, sc_potential, noise_std):
    """Simulate V1-V4 voltages on the FIELDS double-probe.

    Args:
        t: Time grid, shape (N,), seconds.
        e_true: True 2-component electric field in shield plane, shape (N, 2), V/m.
        baseline_12: Effective tip-to-tip baseline of V1-V2 pair, meters.
        baseline_34: Effective tip-to-tip baseline of V3-V4 pair, meters.
        sc_potential: Time-varying spacecraft floating potential, shape (N,), volts.
        noise_std: Per-sensor uncorrelated voltage noise standard deviation, volts.

    Returns:
        V: Sensor voltages (V1, V2, V3, V4), shape (N, 4), volts.
    """
    N = len(t)
    rng = np.random.default_rng(42)
    # Voltage at each tip = -E . r_tip + V_sc + noise (sign convention: tip is at +baseline/2)
    V1 = -e_true[:, 0] * (+baseline_12 / 2.0) + sc_potential + rng.normal(0, noise_std, N)
    V2 = -e_true[:, 0] * (-baseline_12 / 2.0) + sc_potential + rng.normal(0, noise_std, N)
    V3 = -e_true[:, 1] * (+baseline_34 / 2.0) + sc_potential + rng.normal(0, noise_std, N)
    V4 = -e_true[:, 1] * (-baseline_34 / 2.0) + sc_potential + rng.normal(0, noise_std, N)
    return np.stack([V1, V2, V3, V4], axis=1)


def reconstruct_efield_from_probes(V, baseline_12, baseline_34):
    """Reconstruct shield-plane E-field from V1-V4 differential measurements.

    Implements E_{12} = (V_1 - V_2)/L_{12} and E_{34} = (V_3 - V_4)/L_{34} per Bale (2016) Sect. 2.2.2.
    Common-mode spacecraft potential cancels exactly under ideal symmetric conditions.

    Args:
        V: Sensor voltages, shape (N, 4), volts.
        baseline_12: Baseline of V1-V2, meters.
        baseline_34: Baseline of V3-V4, meters.

    Returns:
        E: Reconstructed in-plane E-field, shape (N, 2), V/m.
    """
    e12 = -(V[:, 0] - V[:, 1]) / baseline_12  # sign: E points from + to -
    e34 = -(V[:, 2] - V[:, 3]) / baseline_34
    return np.stack([e12, e34], axis=1)

In [ ]:
# Build a synthetic Alfvenic E-field signal at SPP perihelion plasma
# Table 2 at 10 R_s: |E_c| = v_sw * dB_A ~ 100 mV/m, v_sw = 210 km/s, v_A = 500 km/s
# 합성 알펜 E-field 신호 (Table 2, 10 R_s)
fs = 1500.0           # sampling rate, Hz (well above proton cyclotron 32 Hz)
T_total = 4.0         # signal duration, seconds (~5 NY seconds)
N = int(fs * T_total)
t = np.arange(N) / fs

# True E-field: 50 mV/m amplitude, mostly in 1-50 Hz inertial-range turbulence band
# 진짜 E-field: 50 mV/m, 1-50 Hz 관성영역 난류 대역
rng = np.random.default_rng(0)
white = rng.normal(size=(N, 2))
# Apply f^-5/6 amplitude (i.e. f^-5/3 power) shaping using a Butterworth band
sos = signal.butter(4, [1.0, 50.0], btype='band', fs=fs, output='sos')
shaped = signal.sosfiltfilt(sos, white, axis=0)
e_true = 0.05 * shaped / np.std(shaped) * 0.05 / 0.05  # rescale to ~50 mV/m RMS
e_true *= 0.05 / np.std(e_true)

# Common-mode spacecraft floating potential drifts: ~25 V mean, ~5 V slow drift
# 공통모드 우주선 부유 전위 (~25 V 평균, ~5 V 느린 변동)
sc_phi = -25.0 + 5.0 * np.sin(2 * np.pi * 0.3 * t) + 1.0 * np.cos(2 * np.pi * 0.07 * t)

# Antenna baselines: V1-V2 ~ 7 m tip-to-tip (whip + stub geometry)
# 안테나 기준선: V1-V2 ~ 7 m
L12 = 7.0
L34 = 7.0

# Per-sensor uncorrelated noise: ~10 mV RMS at LF preamp
# 센서별 비상관 잡음: LF preamp ~10 mV RMS
noise = 0.010

V = simulate_fields_voltages(t, e_true, L12, L34, sc_phi, noise)
E_rec = reconstruct_efield_from_probes(V, L12, L34)

print(f"True E_x RMS:        {np.std(e_true[:, 0])*1e3:.2f} mV/m")
print(f"True E_y RMS:        {np.std(e_true[:, 1])*1e3:.2f} mV/m")
print(f"Recovered E_x RMS:   {np.std(E_rec[:, 0])*1e3:.2f} mV/m")
print(f"Recovered E_y RMS:   {np.std(E_rec[:, 1])*1e3:.2f} mV/m")
print(f"Sensor RMS (V1):     {np.std(V[:, 0]):.3f} V (dominated by SC potential)")
print(f"Common-mode rejection: SC drift {np.std(sc_phi):.2f} V suppressed to ~{np.std(E_rec[:,0])*L12*1e3:.1f} mV residual")

In [ ]:
# Visualize raw sensor voltages vs reconstructed E-field
# 원시 센서 전압 대 재구성된 E-field 시각화
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(t, V[:, 0], label='V1', alpha=0.8)
axes[0].plot(t, V[:, 1], label='V2', alpha=0.8)
axes[0].plot(t, sc_phi, 'k--', label='SC potential', linewidth=1.2)
axes[0].set_ylabel('Probe voltage [V]')
axes[0].set_title('Raw V1, V2 (dominated by spacecraft potential) / 원시 V1, V2 (우주선 전위가 지배적)')
axes[0].legend(loc='upper right')
axes[0].grid(alpha=0.3)

axes[1].plot(t, e_true[:, 0]*1e3, 'C0-', label='True E_12', linewidth=1.0)
axes[1].plot(t, E_rec[:, 0]*1e3, 'C3--', label='Reconstructed E_12', linewidth=0.8, alpha=0.8)
axes[1].set_ylabel('E_12 [mV/m]')
axes[1].set_title('Component 1: E_12 = (V1 − V2) / L_12 / 성분 1: E_12')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

axes[2].plot(t, e_true[:, 1]*1e3, 'C0-', label='True E_34', linewidth=1.0)
axes[2].plot(t, E_rec[:, 1]*1e3, 'C3--', label='Reconstructed E_34', linewidth=0.8, alpha=0.8)
axes[2].set_ylabel('E_34 [mV/m]')
axes[2].set_xlabel('Time [s]')
axes[2].set_title('Component 2: E_34 = (V3 − V4) / L_34 / 성분 2: E_34')
axes[2].legend(loc='upper right')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Result interpretation / 결과 해석**: The differential combination $V_1 - V_2$ exactly cancels the ~25 V spacecraft floating potential drift, leaving only the true E-field signal plus uncorrelated sensor noise reduced by $\sqrt{2}/L$. The reconstructed RMS matches the input within sensor-noise-floor limits.

차동 조합 $V_1 - V_2$는 ~25 V의 우주선 부유 전위 변동을 정확히 상쇄하여 진짜 E-field 신호와 $\sqrt{2}/L$로 감쇠된 비상관 센서 잡음만 남긴다. 재구성된 RMS는 센서 잡음 한계 이내에서 입력과 일치한다.

## Part 2: Magnetometer Transfer Functions / 자력계 전달 함수

**Search-coil magnetometer (SCM)**: An inductive sensor obeying Faraday's law,
$$
V_{out}(\omega) = j\omega N A \mu_{eff} B(\omega) - R I(\omega) - j\omega L I(\omega)
$$
After preamp loading, the magnitude transfer function (V/T) follows a single-pole high-pass at low frequency and rolls off above an LC resonance.

Search-coil은 패러데이 법칙을 따르는 유도 센서이다. 저주파에서 단일극 고주파 통과, LC 공진 위에서 감소.

**Fluxgate magnetometer (MAG)**: Saturable-core flat response from DC to ~140 Hz with a low-pass roll-off above the drive frequency.

Fluxgate는 DC ~ ~140 Hz에서 평탄한 응답, drive 주파수 위에서 저주파 통과 감소.

We model the SCM dual-band (ELF/VLF and LF/MF) curves of Fig. 12 and compute the noise floor in nT/√Hz.

Fig. 12의 SCM 이중 대역 곡선을 모사하고 노이즈 한계를 nT/√Hz 단위로 계산.

In [ ]:
def search_coil_response(f_hz, gain_at_resonance, f_resonance, f_low_corner, q_factor=2.0):
    """Model search-coil voltage response as a function of frequency.

    The response is a band-pass: rises as omega below resonance, peaks at f_resonance,
    and rolls off at higher frequencies. Implements a damped harmonic oscillator with
    quality Q.

    Args:
        f_hz: Frequency grid, Hz.
        gain_at_resonance: Voltage-per-Tesla at peak (V/T).
        f_resonance: Resonance frequency, Hz.
        f_low_corner: Low-frequency corner where response rolls off as omega, Hz.
        q_factor: Quality factor of the resonance.

    Returns:
        Magnitude of transfer function |H(f)|, V/T.
    """
    omega = 2.0 * np.pi * f_hz
    omega_r = 2.0 * np.pi * f_resonance
    omega_l = 2.0 * np.pi * f_low_corner
    # Numerator scales with omega (Faraday); denominator gives resonance + roll-off
    num = (omega / omega_l)
    denom = np.sqrt((1.0 - (omega / omega_r) ** 2) ** 2 + (omega / (q_factor * omega_r)) ** 2)
    base = num / denom
    # Normalize so peak equals gain_at_resonance
    base = base / np.max(base) * gain_at_resonance
    return base


def fluxgate_response(f_hz, gain_dc, f_cutoff):
    """Model fluxgate magnetometer response with flat DC gain and 1st-order low-pass.

    Args:
        f_hz: Frequency grid, Hz.
        gain_dc: DC gain (V/T).
        f_cutoff: -3 dB cutoff frequency, Hz.

    Returns:
        Magnitude of transfer function, V/T.
    """
    return gain_dc / np.sqrt(1.0 + (f_hz / f_cutoff) ** 2)

In [ ]:
# Frequency grid spanning DC ~ 1 MHz / DC ~ 1 MHz 주파수 격자
f = np.logspace(0, 6, 600)  # 1 Hz to 1 MHz

# SCM ELF/VLF curve: peak ~5 kHz, low corner ~10 Hz, gain ~1e7 V/T
# (from Fig. 12 "on the left" curve)
H_scm_elf = search_coil_response(f, gain_at_resonance=2e7,
                                 f_resonance=5.0e3,
                                 f_low_corner=10.0,
                                 q_factor=2.5)

# SCM LF/MF curve: peak ~50-100 kHz, low corner ~1 kHz, gain ~1e6 V/T
# (from Fig. 12 "on the right" curve, lower gain by ~50 dB)
H_scm_lf = search_coil_response(f, gain_at_resonance=1e5,
                                f_resonance=1.0e5,
                                f_low_corner=1.0e3,
                                q_factor=2.0)

# Fluxgate: 200 V/T DC gain, -3 dB at 140 Hz / 자속계: DC 200 V/T, -3 dB 140 Hz
H_mag = fluxgate_response(f, gain_dc=200.0, f_cutoff=140.0)

# Convert preamp-input voltage noise (V/sqrt(Hz)) to magnetic noise floor (T/sqrt(Hz))
# 입력 환산 전압 잡음 → 자기장 잡음 한계
preamp_noise_v = 5e-9   # 5 nV/sqrt(Hz)
B_noise_scm_elf = preamp_noise_v / H_scm_elf  # T/sqrt(Hz)
B_noise_scm_lf = preamp_noise_v / H_scm_lf
B_noise_mag = (preamp_noise_v * 200.0) / H_mag  # MAG noise dominated by sensor electronics

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].loglog(f, H_scm_elf, 'C0-', label='SCM ELF/VLF (10 Hz – 50 kHz)')
axes[0].loglog(f, H_scm_lf, 'C3-', label='SCM LF/MF (1 kHz – 1 MHz)')
axes[0].loglog(f, H_mag, 'C2--', label='Fluxgate (DC – 140 Hz)')
axes[0].set_xlabel('Frequency [Hz]')
axes[0].set_ylabel('|H(f)| [V/T]')
axes[0].set_title('Magnetometer transfer functions / 자력계 전달 함수')
axes[0].legend(loc='lower center')
axes[0].grid(which='both', alpha=0.3)
axes[0].set_xlim(1, 1e6)

axes[1].loglog(f, B_noise_scm_elf * 1e9, 'C0-', label='SCM ELF/VLF noise floor')
axes[1].loglog(f, B_noise_scm_lf * 1e9, 'C3-', label='SCM LF/MF noise floor')
axes[1].loglog(f, B_noise_mag * 1e9, 'C2--', label='Fluxgate noise floor')
# Expected turbulence at 10 R_s: dB^2 ~ 10^6.1 nT^2/Hz at f_i=100 Hz
f_grid = np.logspace(-1, 4, 200)
turb_psd = 1e8.1 * 10**-2 * np.where(f_grid < 100, f_grid/100, (f_grid/100)**(-5/3))  # nT^2/Hz scaled
turb_amp = np.sqrt(turb_psd)
axes[1].loglog(f_grid, turb_amp, 'k:', linewidth=1.5, label='Inertial turb @ 10 R_s')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Magnetic noise [nT/√Hz]')
axes[1].set_title('Magnetic-field noise floor / 자기장 잡음 한계')
axes[1].legend(loc='upper right')
axes[1].grid(which='both', alpha=0.3)
axes[1].set_xlim(1, 1e6)
axes[1].set_ylim(1e-7, 1e2)

plt.tight_layout()
plt.show()

**Result interpretation / 결과 해석**: The SCM ELF/VLF curve peaks near 5 kHz, the LF/MF channel peaks near 100 kHz, with the two curves overlapping at the 1 kHz crossover (Fig. 12 of the paper). The fluxgate has flat DC gain and -3 dB at 140 Hz. The magnetic-field noise floor (right panel) is below the expected inertial-range turbulence at 10 R_s by 1-2 decades over the science band — confirming the SCM and MAG sensitivity is sufficient to resolve the predicted Alfvén/MHD turbulence down to perihelion.

SCM ELF/VLF는 ~5 kHz, LF/MF는 ~100 kHz에서 정점에 도달하며 1 kHz 교차에서 겹친다(논문 Fig. 12). Fluxgate는 평탄한 DC 이득과 -3 dB at 140 Hz. 자기장 잡음 한계(우측)는 10 R_s에서 예측되는 관성영역 난류보다 1-2 십진 아래에 있어 — SCM/MAG 감도가 근일점까지 알펜·MHD 난류를 분해하기에 충분함을 확인.

## Part 3: DC-to-MHz Wave Dispersion at SPP Perihelion / SPP 근일점 DC ~ MHz 파동 분산

**Concept / 개념**: FIELDS spans DC to ~20 MHz to capture the entire spectrum from MHD Alfvén waves through ion-cyclotron and whistler regimes up to Langmuir/electron plasma waves. We compute and overlay the dispersion relations at SPP perihelion plasma parameters from Table 2.

FIELDS는 DC ~ 20 MHz에 걸쳐 MHD 알펜파부터 이온-사이클로트론, 휘슬러, 랭뮤어/전자 플라즈마 파까지 전 스펙트럼을 포착한다. Table 2 SPP 근일점 매개변수에서 분산 관계를 계산하여 중첩한다.

Dispersion relations / 분산 관계:
- **Alfvén wave** (MHD): $\omega = k v_A$
- **Fast magnetosonic / 빠른 자기음파**: $\omega^2 = k^2 (v_A^2 + c_s^2)$ (parallel approximation)
- **Whistler** (low-frequency): $\omega \approx \Omega_e (k c / \omega_{pe})^2 / [1 + (k c / \omega_{pe})^2]$
- **Langmuir / 랭뮤어**: $\omega^2 = \omega_{pe}^2 + 3 k^2 v_{th,e}^2$

In [ ]:
def plasma_parameters(n_e_cc, T_e_eV, B_nT):
    """Compute fundamental plasma frequencies and lengths.

    Args:
        n_e_cc: Electron density, cm^-3.
        T_e_eV: Electron temperature, eV.
        B_nT: Magnetic field, nT.

    Returns:
        Dictionary with f_pe, f_ce, f_cp, v_A, v_the, debye_m, c_over_wpe (electron inertial length).
    """
    n_e = n_e_cc * 1e6  # m^-3
    T_e = T_e_eV * EV_TO_J
    B = B_nT * 1e-9

    omega_pe = np.sqrt(n_e * QE ** 2 / (EPS0 * ME))
    omega_ce = QE * B / ME
    omega_cp = QE * B / MP
    v_th_e = np.sqrt(T_e / ME)
    n_proton = n_e  # quasineutral
    rho = n_proton * MP
    v_A = B / np.sqrt(MU0 * rho)
    debye = np.sqrt(EPS0 * T_e / (n_e * QE ** 2))
    c_wpe = C_LIGHT / omega_pe
    c_wpi = C_LIGHT / np.sqrt(n_proton * QE ** 2 / (EPS0 * MP))

    return {
        'f_pe': omega_pe / (2 * np.pi),
        'f_ce': omega_ce / (2 * np.pi),
        'f_cp': omega_cp / (2 * np.pi),
        'v_A': v_A,
        'v_th_e': v_th_e,
        'debye_m': debye,
        'c_over_wpe': c_wpe,
        'c_over_wpi': c_wpi,
        'omega_pe': omega_pe,
        'omega_ce': omega_ce,
        'omega_cp': omega_cp,
    }

In [ ]:
# Three reference distances from Table 2
scenarios = {
    '10 R_s (perihelion)': dict(n_e_cc=7000, T_e_eV=85, B_nT=2000),
    '55 R_s': dict(n_e_cc=120, T_e_eV=25, B_nT=70),
    '1 AU': dict(n_e_cc=7, T_e_eV=8, B_nT=6),
}

for name, params in scenarios.items():
    p = plasma_parameters(**params)
    print(f"{name}:")
    print(f"  f_pe = {p['f_pe']/1e3:.1f} kHz   f_ce = {p['f_ce']/1e3:.1f} kHz   f_cp = {p['f_cp']:.1f} Hz")
    print(f"  v_A = {p['v_A']/1e3:.0f} km/s    v_th,e = {p['v_th_e']/1e3:.0f} km/s")
    print(f"  Debye = {p['debye_m']*1e3:.2f} mm   c/omega_pe = {p['c_over_wpe']:.2f} m   c/omega_pi = {p['c_over_wpi']:.0f} m")
    print()

In [ ]:
# Dispersion relations at perihelion / 근일점 분산 관계
p = plasma_parameters(n_e_cc=7000, T_e_eV=85, B_nT=2000)

# Wavenumber grid spanning convected scales (k ~ 1/lambda)
k = np.logspace(-7, 2, 500)  # m^-1

# 1) Alfven wave / 알펜파
omega_alfven = k * p['v_A']

# 2) Fast magnetosonic / 빠른 자기음파 (assume v_A >> c_s for cold plasma at perihelion)
c_s = np.sqrt(85 * EV_TO_J / MP)
omega_fast = k * np.sqrt(p['v_A'] ** 2 + c_s ** 2)

# 3) Whistler dispersion / 휘슬러 분산
kc_wpe2 = (k * p['c_over_wpe']) ** 2
omega_whistler = p['omega_ce'] * kc_wpe2 / (1.0 + kc_wpe2)

# 4) Langmuir wave / 랭뮤어파
omega_langmuir = np.sqrt(p['omega_pe'] ** 2 + 3.0 * (k * p['v_th_e']) ** 2)

# 5) EM wave (vacuum) / 진공 EM
omega_em = k * C_LIGHT

fig, ax = plt.subplots(figsize=(10, 7))
ax.loglog(k, omega_alfven / (2 * np.pi), 'C0-', label='Alfvén  ω = k v_A', linewidth=1.5)
ax.loglog(k, omega_fast / (2 * np.pi), 'C0--', label='Fast magnetosonic', linewidth=1.0, alpha=0.7)
ax.loglog(k, omega_whistler / (2 * np.pi), 'C3-', label='Whistler', linewidth=1.5)
ax.loglog(k, omega_langmuir / (2 * np.pi), 'C2-', label='Langmuir  ω² = ω_pe² + 3 k² v_th,e²', linewidth=1.5)
ax.loglog(k, omega_em / (2 * np.pi), 'k:', label='EM (vacuum)', linewidth=1.0, alpha=0.6)

# Frequency reference lines / 주파수 기준선
ax.axhline(p['f_cp'], color='gray', linestyle=':', alpha=0.7)
ax.axhline(p['f_ce'], color='gray', linestyle=':', alpha=0.7)
ax.axhline(p['f_pe'], color='gray', linestyle=':', alpha=0.7)
ax.text(2e-7, p['f_cp']*1.2, 'f_cp = 32 Hz', fontsize=9)
ax.text(2e-7, p['f_ce']*1.2, 'f_ce = 60 kHz', fontsize=9)
ax.text(2e-7, p['f_pe']*1.2, 'f_pe = 750 kHz', fontsize=9)

# FIELDS measurement bands / FIELDS 측정 대역
ax.axhspan(0.001, 50e3, facecolor='C2', alpha=0.05)  # MAG range
ax.axhspan(50e3, 1e6, facecolor='C0', alpha=0.05)    # SCM/E LF/MF
ax.axhspan(1e6, 20e6, facecolor='C3', alpha=0.05)    # RFS
ax.text(1.5e1, 1e2, 'MAG band', color='C2', fontsize=9)
ax.text(1.5e1, 2e5, 'SCM/E AC band', color='C0', fontsize=9)
ax.text(1.5e1, 5e6, 'RFS band (10 kHz – 20 MHz)', color='C3', fontsize=9)

ax.set_xlabel('Wavenumber k [1/m]')
ax.set_ylabel('Frequency f = ω/2π [Hz]')
ax.set_title('Wave dispersion at SPP perihelion (10 R_s) / SPP 근일점(10 R_s)에서 파동 분산')
ax.legend(loc='lower right', fontsize=10)
ax.grid(which='both', alpha=0.3)
ax.set_xlim(1e-7, 1e2)
ax.set_ylim(1e-2, 1e8)
plt.tight_layout()
plt.show()

**Result interpretation / 결과 해석**: At 10 R_s, the Alfvén branch transitions into the whistler branch at k c/ω_pe ~ 1 (i.e. k ~ 1/(0.3 m) = 3 m^-1 — a few proton inertial lengths) and into the ion-cyclotron resonance at f_cp = 32 Hz. The whistler branch asymptotes to f_ce = 60 kHz. Langmuir waves sit at f_pe = 750 kHz. The shaded bands show the MAG (DC – 50 kHz, green), SCM/E AC (~5 kHz – 1 MHz, blue), and RFS (10 kHz – 20 MHz, red) coverage — together they span every important branch in the inner heliosphere from MHD to plasma oscillations.

10 R_s에서 알펜 분기는 k c/ω_pe ~ 1 (즉 k ~ 1/(0.3 m) = 3 m^-1, 양성자 관성거리 몇 배)에서 휘슬러 분기로 전이하고, f_cp = 32 Hz에서 이온-사이클로트론 공명으로 흡수된다. 휘슬러 분기는 f_ce = 60 kHz에 점근. 랭뮤어 파는 f_pe = 750 kHz. 음영은 MAG(DC ~ 50 kHz, 초록), SCM/E AC(~5 kHz ~ 1 MHz, 파랑), RFS(10 kHz ~ 20 MHz, 빨강) 대역 — 합쳐서 내부 태양권의 MHD부터 플라즈마 진동까지 모든 핵심 분기를 포함.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Double-probe E-field | V1-V4 differential at TPS base, V5 monopole on boom; common-mode SC potential subtraction / TPS 기저의 V1-V4 차동, boom의 V5 monopole; 공통모드 SC 전위 제거 | Solar Orbiter RPW Antennas (2020), JUICE RPWI (2023) — same heritage Berkeley/LESIA design / Solar Orbiter RPW, JUICE RPWI — 동일 헤리티지 |
| Dual-band magnetometer | Fluxgate (DC – 140 Hz, 16-bit ±65,536 nT) + Search-coil (10 Hz – 1 MHz, 160 dB) on 3.5 m boom / 3.5 m boom의 fluxgate(DC ~ 140 Hz, 16-bit ±65,536 nT) + search-coil(10 Hz ~ 1 MHz, 160 dB) | Same fluxgate lineage on MAVEN, RBSP, JUICE; search-coil heritage on Solar Orbiter RPW SCM, JUICE RPWI SCM / MAVEN, RBSP, JUICE의 동일 fluxgate 계보; Solar Orbiter, JUICE의 search-coil 헤리티지 |
| Wave-mode coverage | DC to 20 MHz: MAG → SCM/E → RFS chain capturing Alfvén → ion-cyclotron → whistler → Langmuir / DC ~ 20 MHz: MAG → SCM/E → RFS 사슬로 알펜 → 이온-사이클로트론 → 휘슬러 → 랭뮤어 포착 | Inheritance for SUNRISE-3 (high-resolution remote), MUSE (EUV), and proposed Solar Polar mission / SUNRISE-3, MUSE, 제안된 Solar Polar 미션의 계승 |
| Picket-fence EMC + NY second | All DC-DC at multiples of 150 kHz; cadence = power-of-2 of 1 NYsec ≈ 0.874 s / 모든 DC-DC가 150 kHz 배수; cadence = 1 NYsec ≈ 0.874 s의 2 거듭제곱 | Solar Orbiter and Europa Clipper adopted similar synchronized chopping / Solar Orbiter, Europa Clipper도 유사한 동기 chopping 채택 |
| Burst memory + CBS | DBM holds 3.5 s × 6 channels at 150 kSa/s, ranked competitively; CBS aggregates DFB+TDS+RFS+SWEAP at 4×/NYsec / DBM이 150 kSa/s × 6 채널 × 3.5 s 보관, 경쟁 평가; CBS는 DFB+TDS+RFS+SWEAP을 4×/NYsec로 통합 | Same architecture in MMS, JUICE — quality-ranked event capture is now standard / MMS, JUICE의 동일 아키텍처 — 품질 평가 이벤트 포착이 표준 |

**Final note / 마무리**: Bale et al. (2016) is more than an instrument paper — it is the Rosetta Stone for interpreting every PSP fields/waves measurement that has and will appear. Future near-Sun missions (e.g. ESA Solar Polar, JAXA SolarPlus) will continue to leverage its proven design.

Bale 외 (2016)는 단순한 기기 논문이 아니라 PSP의 모든 fields/waves 측정을 해석하는 로제타석이다. 미래의 근태양 미션(ESA Solar Polar, JAXA SolarPlus 등)도 이 입증된 설계를 계승할 것이다.